In [2]:
import torch
from torch.profiler import profile, ProfilerActivity
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from tabulate import tabulate

test_x = np.load('test_x.npy').astype(np.float32)
test_y = np.load('test_y.npy').astype(np.float32)
test_dataset = TensorDataset(torch.tensor(test_x).float(), torch.tensor(test_y).long())

train_x = np.load("train_x.npy").astype(np.float32)
train_y = np.load("train_y.npy").astype(np.float32)
train_dataset = TensorDataset(torch.tensor(train_x).float(), torch.tensor(train_y).long())

batch_size = 100
n_iters = 3000
num_epochs = n_iters / (len(train_dataset) / batch_size)
num_epochs = int(num_epochs)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

class FeedforwardNeuralNetModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FeedforwardNeuralNetModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.bn3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        return out

input_dim = test_x.shape[1]
hidden_dim = 64
output_dim = len(torch.unique(torch.tensor(test_y)))

model = FeedforwardNeuralNetModel(input_dim, hidden_dim, output_dim)
torch.save(model.state_dict(), "model.pth")

model.load_state_dict(torch.load("model.pth"))
model = model.to("cpu")
model.eval()

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True
) as prof:
    for i, (inputs, _) in enumerate(test_loader):
        if i >= 100:
            break
        model(inputs)

results = [
    [m.key, f"{m.cpu_time_total:.2f} ms", f"{m.self_cpu_memory_usage / 1024:.2f} KB"]
    for m in prof.key_averages() if any(x in m.key.lower() for x in ["relu", "batch_norm", "linear"])
]

print(tabulate(results, headers=["Module", "CPU Time (ms)", "Memory (KB)"], tablefmt="grid"))

c:\ProgramData\anaconda3\lib\site-packages\torch\autograd\profiler.py:160: UserWarning: CUDA is not available, disabling CUDA profiling
  warn("CUDA is not available, disabling CUDA profiling")


+------------------------------+-----------------+---------------+
| Module                       | CPU Time (ms)   | Memory (KB)   |
+==============================+=================+===============+
| aten::linear                 | 527640.00 ms    | 0.00 KB       |
+------------------------------+-----------------+---------------+
| aten::batch_norm             | 27639.00 ms     | 0.00 KB       |
+------------------------------+-----------------+---------------+
| aten::_batch_norm_impl_index | 24583.00 ms     | 0.00 KB       |
+------------------------------+-----------------+---------------+
| aten::native_batch_norm      | 18173.00 ms     | -150.00 KB    |
+------------------------------+-----------------+---------------+
| aten::relu                   | 16056.00 ms     | 0.00 KB       |
+------------------------------+-----------------+---------------+


In [3]:
import torch
from torch.profiler import profile, ProfilerActivity
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from tabulate import tabulate

test_x = np.load('test_x.npy').astype(np.float32)
test_y = np.load('test_y.npy').astype(np.float32)
test_dataset = TensorDataset(torch.tensor(test_x).float(), torch.tensor(test_y).long())

train_x = np.load("train_x.npy").astype(np.float32)
train_y = np.load("train_y.npy").astype(np.float32)
train_dataset = TensorDataset(torch.tensor(train_x).float(), torch.tensor(train_y).long())

batch_size = 100
n_iters = 3000
num_epochs = n_iters / (len(train_dataset) / batch_size)
num_epochs = int(num_epochs)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

class FeedforwardNeuralNetModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, activation_fn):
        super(FeedforwardNeuralNetModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.act1 = activation_fn()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.act2 = activation_fn()
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.act3 = activation_fn()
        self.fc4 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out = self.act2(out)
        out = self.fc3(out)
        out = self.bn3(out)
        out = self.act3(out)
        out = self.fc4(out)
        return out

input_dim = test_x.shape[1]
hidden_dim = 64
output_dim = len(torch.unique(torch.tensor(test_y)))

activations = {
    "ReLU": nn.ReLU,
    "Tanh": nn.Tanh,
    "Sigmoid": nn.Sigmoid,
    "GeLU": nn.GELU
}

with open("profiling_results.txt", "w") as file:
    for name, activation_fn in activations.items():
        file.write(f"\nRunning for activation function: {name}\n")
        model = FeedforwardNeuralNetModel(input_dim, hidden_dim, output_dim, activation_fn)
        model.eval()

        with profile(
            activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
            profile_memory=True,
            record_shapes=True
        ) as prof:
            with torch.no_grad():
                for i, (inputs, _) in enumerate(test_loader):
                    if i >= 100:
                        break
                    outputs = model(inputs)

        file.write(f"Profiling results for {name}:\n")
        file.write(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))
        file.write("\n" + "="*80 + "\n")